<a href="https://colab.research.google.com/github/ngyxntthaoo/IS6404.CH201-Phan-tich-du-lieu-nang-cao/blob/main/Lab04c_Lec06_TimeSeries_ML_Forecasting_PTDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4c (Lec06): Time Series Machine Learning - Forecasting
**Course:** IS6404 - Advanced Data Analysis, UIT, VNUHCM  
**Lecturer:** Dr. Tran Hung Nghiep, 2026  

---

Notebook này minh họa pipeline chuẩn cho **time series forecasting** theo hướng thực dụng:

- Baseline cổ điển: naive / seasonal naive / ARIMA
- ML-based forecasting: lag features + ML model
- So sánh mô hình theo **time-based split**
- Phân tích lỗi và những bẫy phổ biến trong forecasting

> Trọng tâm của bài lab này: **temporal order matters**.  
> Với time series, split, leakage, seasonality, và horizon forecasting thường quan trọng hơn việc đổi model phức tạp.


## 0. Setup & reproducibility

In [ ]:
# Nếu chạy trên Colab và thiếu statsmodels, mở comment:
# !pip -q install statsmodels

import os, json, sys, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("python:", sys.version)
print("platform:", platform.platform())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

warnings.filterwarnings("ignore")


## 1. Problem formulation

Bài toán hôm nay: **forecasting một chuỗi thời gian đơn biến**.

Ví dụ điển hình:
- doanh số theo ngày/tuần/tháng
- traffic website
- electricity demand
- sensor readings
- giá tài sản

### Câu hỏi gợi ý
1. Forecasting target là gì?
2. Forecast horizon là gì? (1 ngày tới, 7 ngày tới, 1 tháng tới?)
3. Forecast origin là thời điểm nào?
4. Metric chính là gì? MAE, RMSE, MAPE, hay sMAPE? Vì sao?


## 2. Tạo dữ liệu time series mô phỏng

Để notebook chạy ổn định, ta tạo một chuỗi có:
- trend
- weekly seasonality
- yearly seasonality nhẹ
- noise
- một vài shock events

Bối cảnh giả định: daily sales.


In [ ]:
date_index = pd.date_range(start="2020-01-01", end="2023-12-31", freq="D")
n = len(date_index)
t = np.arange(n)

trend = 100 + 0.03 * t
weekly = 12 * np.sin(2 * np.pi * t / 7)
yearly = 8 * np.sin(2 * np.pi * t / 365.25)
noise = np.random.normal(0, 4, size=n)

y = trend + weekly + yearly + noise

# Add a few shocks
shock_days = ["2020-07-15", "2021-11-20", "2022-06-10", "2023-03-05"]
for d in shock_days:
    idx = np.where(date_index == pd.Timestamp(d))[0]
    if len(idx) > 0:
        y[idx[0]: idx[0] + 5] += np.array([20, 15, 10, 5, 0])

df = pd.DataFrame({
    "ds": date_index,
    "y": y
})

df.head()


In [ ]:
df.set_index("ds")["y"].plot(figsize=(12,4), title="Synthetic Daily Sales Time Series")
plt.show()

df.describe()


## 3. Time-based split (bắt buộc)

Với forecasting, **không dùng random split**.

Ta dùng:
- train: dữ liệu quá khứ
- validation: một đoạn thời gian ngay sau train
- test: đoạn thời gian cuối cùng

### Câu hỏi gợi ý
1. Vì sao random split là sai trong forecasting?
2. Nếu bài toán triển khai thực tế là dự báo tương lai, split nào phản ánh đúng deploy?


In [ ]:
# Example split:
# train: 2020-01-01 -> 2022-12-31
# val:   2023-01-01 -> 2023-06-30
# test:  2023-07-01 -> 2023-12-31

train_df = df[df["ds"] < "2023-01-01"].copy()
val_df = df[(df["ds"] >= "2023-01-01") & (df["ds"] < "2023-07-01")].copy()
test_df = df[df["ds"] >= "2023-07-01"].copy()

len(train_df), len(val_df), len(test_df)


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(train_df["ds"], train_df["y"], label="train")
plt.plot(val_df["ds"], val_df["y"], label="val")
plt.plot(test_df["ds"], test_df["y"], label="test")
plt.legend()
plt.title("Time-based split")
plt.show()


## 4. Evaluation metrics for forecasting

Ta dùng:
- MAE: dễ hiểu, robust
- RMSE: phạt lỗi lớn mạnh hơn
- MAPE: dễ giải thích theo %, nhưng cẩn thận khi y gần 0

Ngoài ra, cần nhìn:
- residuals
- lỗi theo thời gian
- lỗi theo seasonality


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100)

def eval_forecast(y_true, y_pred, prefix="val"):
    return {
        f"{prefix}_MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}_RMSE": rmse(y_true, y_pred),
        f"{prefix}_MAPE": mape(y_true, y_pred),
    }


## 5. Baseline 1: Naive forecast

Naive:
- dự báo ngày tiếp theo bằng giá trị gần nhất

Đây là baseline cực quan trọng trong forecasting.
Nếu mô hình phức tạp không thắng naive rõ ràng, có thể có vấn đề.


In [ ]:
# Forecast val using last observed point from train, recursively as a flat forecast
last_train_value = train_df["y"].iloc[-1]
val_pred_naive = np.repeat(last_train_value, len(val_df))

eval_forecast(val_df["y"], val_pred_naive, prefix="val")


## 6. Baseline 2: Seasonal naive forecast

Với daily data có weekly seasonality:
- dự báo hôm nay bằng giá trị của 7 ngày trước

Seasonal naive thường là baseline rất mạnh.


In [ ]:
def seasonal_naive_predict(history_values, horizon, season_length=7):
    history_values = list(history_values)
    preds = []
    for i in range(horizon):
        preds.append(history_values[-season_length + (i % season_length)])
    return np.array(preds)

val_pred_snaive = seasonal_naive_predict(train_df["y"].values, len(val_df), season_length=7)
eval_forecast(val_df["y"], val_pred_snaive, prefix="val")


## 7. Baseline 3: ARIMA / SARIMAX

ARIMA là mô hình cổ điển quan trọng cho time series.
Trong thực hành:
- tốt khi cấu trúc chuỗi tương đối đều
- mạnh như baseline nghiên cứu
- dễ diễn giải hơn nhiều mô hình deep learning

Ở đây ta dùng SARIMAX để mô hình hóa seasonality tuần.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Một cấu hình nhỏ để notebook chạy ổn:
# order=(2,1,2), seasonal_order=(1,0,1,7)
sarimax = SARIMAX(
    train_df["y"],
    order=(2,1,2),
    seasonal_order=(1,0,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_fit = sarimax.fit(disp=False)

val_pred_arima = sarimax_fit.forecast(steps=len(val_df))
eval_forecast(val_df["y"], val_pred_arima, prefix="val")


## 8. ML-based forecasting: lag features + tree-based model

Ý tưởng:
- biến time series thành supervised learning
- tạo lag features, rolling statistics, calendar features
- train model như tabular ML

Đây là hướng rất mạnh trong thực hành business forecasting.


In [ ]:
def make_ts_features(df_in):
    df_feat = df_in.copy()
    df_feat = df_feat.sort_values("ds").reset_index(drop=True)

    # Lag features
    for lag in [1, 2, 3, 7, 14, 28]:
        df_feat[f"lag_{lag}"] = df_feat["y"].shift(lag)

    # Rolling features
    for window in [7, 14, 28]:
        df_feat[f"roll_mean_{window}"] = df_feat["y"].shift(1).rolling(window).mean()
        df_feat[f"roll_std_{window}"] = df_feat["y"].shift(1).rolling(window).std()

    # Calendar features
    df_feat["dayofweek"] = df_feat["ds"].dt.dayofweek
    df_feat["month"] = df_feat["ds"].dt.month
    df_feat["dayofyear"] = df_feat["ds"].dt.dayofyear

    return df_feat

df_feat = make_ts_features(df)
df_feat.head(10)


In [ ]:
# Split again after feature generation, but NEVER use future information
train_feat = df_feat[df_feat["ds"] < "2023-01-01"].copy()
val_feat = df_feat[(df_feat["ds"] >= "2023-01-01") & (df_feat["ds"] < "2023-07-01")].copy()
test_feat = df_feat[df_feat["ds"] >= "2023-07-01"].copy()

feature_cols = [c for c in train_feat.columns if c not in ["ds", "y"]]

train_feat = train_feat.dropna().reset_index(drop=True)
val_feat = val_feat.dropna().reset_index(drop=True)
test_feat = test_feat.dropna().reset_index(drop=True)

X_train = train_feat[feature_cols]
y_train = train_feat["y"]
X_val = val_feat[feature_cols]
y_val = val_feat["y"]
X_test = test_feat[feature_cols]
y_test = test_feat["y"]

X_train.shape, X_val.shape, X_test.shape


## 9. Model A (ML-based): Linear regression with lag features

Đây là baseline ML-based đơn giản:
- dễ hiểu
- dễ kiểm tra
- nối logic trực tiếp từ regression buổi trước


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

ridge_ts = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_ts.fit(X_train, y_train)
val_pred_ridge = ridge_ts.predict(X_val)

eval_forecast(y_val, val_pred_ridge, prefix="val")


## 10. Model B (ML-based): Tree boosting with lag features

Với tabularized time series, tree boosting thường rất mạnh.
Ở đây ta dùng HistGradientBoostingRegressor từ sklearn để giữ pipeline gọn.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_ts = HistGradientBoostingRegressor(
    random_state=RANDOM_SEED,
    max_depth=5,
    learning_rate=0.05
)

hgb_ts.fit(X_train, y_train)
val_pred_hgb = hgb_ts.predict(X_val)

eval_forecast(y_val, val_pred_hgb, prefix="val")


## 11. So sánh các mô hình trên validation

In [ ]:
val_table = pd.DataFrame([
    {"model": "naive", **eval_forecast(val_df["y"], val_pred_naive, prefix="val")},
    {"model": "seasonal_naive", **eval_forecast(val_df["y"], val_pred_snaive, prefix="val")},
    {"model": "sarimax", **eval_forecast(val_df["y"], val_pred_arima, prefix="val")},
    {"model": "ridge_lag_features", **eval_forecast(y_val, val_pred_ridge, prefix="val")},
    {"model": "hgb_lag_features", **eval_forecast(y_val, val_pred_hgb, prefix="val")},
]).sort_values(by="val_MAE")

val_table

## 12. Visual comparison on validation

Trong forecasting, nhìn đồ thị dự báo rất quan trọng.
Metric không đủ để hiểu hành vi mô hình.


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(val_df["ds"], val_df["y"], label="actual", linewidth=2)
plt.plot(val_df["ds"], val_pred_naive, label="naive", alpha=0.5)
plt.plot(val_df["ds"], val_pred_snaive, label="seasonal_naive", alpha=0.5)
plt.plot(val_feat["ds"], val_pred_ridge, label="ridge_lag", alpha=0.8)
plt.plot(val_feat["ds"], val_pred_hgb, label="hgb_lag", alpha=0.8)
plt.legend()
plt.title("Validation forecast comparison (including Prophet)")
plt.show()

## 13. Feature importance / interpretability cho ML-based forecasting

Với mô hình lag-feature + tree, feature importance cho biết:
- lag feature nào quan trọng
- seasonality ở mức nào
- rolling statistics có hữu ích không


### Model A (Ridge with lag features)

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance for the Ridge model
perm_importance_ridge = permutation_importance(ridge_ts, X_val, y_val, n_repeats=10, random_state=RANDOM_SEED)

imp_df_ridge = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm_importance_ridge.importances_mean
}).sort_values(by="importance", ascending=False)

print("Ridge Model Feature Importance:")
display(imp_df_ridge.head(15))

In [ ]:
top = imp_df_ridge.head(15).iloc[::-1]
plt.figure(figsize=(8,5))
plt.barh(top["feature"], top["importance"])
plt.title("Top feature importances - Ridge time series model")
plt.show()


### Model B (HistGradientBoosting with lag features)

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance
perm_importance = permutation_importance(hgb_ts, X_val, y_val, n_repeats=10, random_state=RANDOM_SEED)

imp_df_hgb = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm_importance.importances_mean
}).sort_values(by="importance", ascending=False)

imp_df_hgb.head(15)

In [ ]:
top = imp_df_hgb.head(15).iloc[::-1]
plt.figure(figsize=(8,5))
plt.barh(top["feature"], top["importance"])
plt.title("Top feature importances - HGB time series model")
plt.show()


## 14. Error analysis

Trong forecasting, cần hỏi:
- mô hình sai ở đoạn nào?
- sai ở peak/trough hay sai toàn cục?
- mô hình có bắt được seasonality không?
- shock events có làm mô hình lệch mạnh không?

### Câu hỏi gợi ý
1. Mô hình nào bắt seasonality tốt nhất?
2. Mô hình nào phản ứng kém với shock events?
3. Nếu business quan tâm peak demand, metric nào nên ưu tiên?


## 15. Final test evaluation

Cần chọn best model trên validation rồi mới đánh giá test.  
Không dùng kết quả đánh giá test để lựa chọn best model.


In [ ]:
best_model_name = val_table.iloc[0]["model"]
print("Best validation model:", best_model_name)


Có thể retrain với toàn bộ dữ liệu train + validation để có model cuối cùng trước khi test.

In [ ]:
# Prepare test predictions consistently
last_train_val = df[df["ds"] < "2023-07-01"]["y"].iloc[-1]
test_pred_naive = np.repeat(last_train_val, len(test_df))

history_until_test = df[df["ds"] < "2023-07-01"]["y"].values
test_pred_snaive = seasonal_naive_predict(history_until_test, len(test_df), season_length=7)

# Refit SARIMAX on train + val
trainval_df = df[df["ds"] < "2023-07-01"].copy()
sarimax_tv = SARIMAX(
    trainval_df["y"],
    order=(2,1,2),
    seasonal_order=(1,0,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_tv_fit = sarimax_tv.fit(disp=False)
test_pred_arima = sarimax_tv_fit.forecast(steps=len(test_df))

# Refit ML models on train+val features
trainval_feat = df_feat[df_feat["ds"] < "2023-07-01"].copy().dropna().reset_index(drop=True)
X_trainval = trainval_feat[feature_cols]
y_trainval = trainval_feat["y"]

ridge_ts_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
ridge_ts_final.fit(X_trainval, y_trainval)
test_pred_ridge = ridge_ts_final.predict(X_test)

hgb_ts_final = HistGradientBoostingRegressor(
    random_state=RANDOM_SEED,
    max_depth=5,
    learning_rate=0.05
)
hgb_ts_final.fit(X_trainval, y_trainval)
test_pred_hgb = hgb_ts_final.predict(X_test)

test_table = pd.DataFrame([
    {"model": "naive", **eval_forecast(test_df["y"], test_pred_naive, prefix="test")},
    {"model": "seasonal_naive", **eval_forecast(test_df["y"], test_pred_snaive, prefix="test")},
    {"model": "sarimax", **eval_forecast(test_df["y"], test_pred_arima, prefix="test")},
    {"model": "ridge_lag_features", **eval_forecast(y_test, test_pred_ridge, prefix="test")},
    {"model": "hgb_lag_features", **eval_forecast(y_test, test_pred_hgb, prefix="test")},
]).sort_values(by="test_MAE")

test_table


## 16. Save results (reproducibility)

In [ ]:
results = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "seed": RANDOM_SEED,
    "val_table": val_table.to_dict(orient="records"),
    "test_table": test_table.to_dict(orient="records"),
    "top_features_ridge": imp_df_ridge.head(20).to_dict(orient="records"),
}

with open(os.path.join(OUTPUT_DIR, "results_lab04c_timeseries.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

val_table.to_csv(os.path.join(OUTPUT_DIR, "val_table_lab04c_timeseries.csv"), index=False)
test_table.to_csv(os.path.join(OUTPUT_DIR, "test_table_lab04c_timeseries.csv"), index=False)
imp_df_ridge.to_csv(os.path.join(OUTPUT_DIR, "feature_importance_lab04c_timeseries.csv"), index=False)

print("Saved: outputs/results_lab04c_timeseries.json, val/test tables, feature importance")


## 17. Prophet: baseline mạnh và dễ sử dụng

- Lag features + ML rất mạnh, nhưng yêu cầu nhiều feature engineering thủ công để kết quả cao hơn.
- Prophet tự động thực hiện feature engineering, dễ dàng có kết quả tốt, có thể cao hơn lag features + ML cơ bản.


In [ ]:
from prophet import Prophet

# Initialize and fit Prophet model on training data
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
prophet_model.fit(train_df.rename(columns={'ds': 'ds', 'y': 'y'}))

# Create a future DataFrame for predictions
future = prophet_model.make_future_dataframe(periods=len(val_df), freq='D', include_history=False)

# Make predictions
forecast = prophet_model.predict(future)

val_pred_prophet = forecast['yhat'].values

# Evaluate Prophet model on validation set
eval_forecast(val_df["y"], val_pred_prophet, prefix="val")


## 18. Short report

Trả lời ngắn gọn:

1. Vì sao random split là sai trong forecasting?
2. Baseline cổ điển nào mạnh nhất: naive, seasonal naive, hay SARIMAX?
3. ML-based model có thắng baseline cổ điển không? Ở metric nào?
4. Lag feature nào quan trọng nhất? Điều đó nói gì về seasonality?
5. Mô hình nào phản ứng kém với shock events?
6. Nếu áp dụng vào đồ án, bạn sẽ ưu tiên:
   - classical forecasting
   - lag-feature + ML
   - hay Prophet?
   Vì sao?

### Bài tập về nhà
- Nộp notebook + outputs đã chạy và trả lời câu hỏi
- Thử một mở rộng (nếu muốn):
  - thay đổi forecast horizon
  - thêm holiday / event features
  - tune SARIMAX
  - tune HGB
  - nếu nhóm làm forecasting thật: thay dataset bằng dữ liệu của nhóm
